# 02 — Grain, Candidate Keys & Duplicates

**Objetivo:** probar hipótesis de granularidad y claves sin asumir que una columna es PK solo porque se llame ID.

Este notebook es un **laboratorio reproducible**. La lógica reutilizable vive en `src/argos/`;
aquí solo se parametriza, ejecuta, inspecciona evidencia y documentan decisiones.

## Preguntas
- ¿Qué representa una fila en cada fuente?
- ¿Qué claves candidatas son realmente únicas?
- ¿Cuántos nulls tienen?
- ¿Cuál es la multiplicidad máxima?

**Interpretación:** una clave no es aceptada por nombre; es aceptada por evidencia + semántica de negocio.

In [1]:
from pathlib import Path
import tomllib
import pandas as pd

from argos.common.paths import ProjectPaths
from argos.config.registry import load_registry, discover_dataset_files

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()
DATASETS, RELATIONSHIPS = load_registry(paths.configs / "datasets.toml")
FILES = discover_dataset_files(paths.bronze, DATASETS)

with (paths.configs / "project.toml").open("rb") as handle:
    PROJECT_CONFIG = tomllib.load(handle)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)

{k: p.name for k, p in FILES.items()}

{'processes': 'SECOP_II_-_Procesos_de_Contratación_20260921.csv',
 'contracts': 'SECOP_II_-_Contratos_Electrónicos_20260921.csv',
 'bidders': 'Proponentes_por_Proceso_SECOP_II_20260921.csv',
 'suppliers': 'SECOP_II_-_Proveedores_Registrados_20260921.csv',
 'additions': 'SECOP_II_-_Adiciones_20260921.csv',
 'sanctions': 'SECOPII_-_Multas_y_Sanciones_20260921.csv',
 'payments': 'SECOP_II_-_Plan_de_pagos_20260921.csv'}

In [2]:
from argos.profiling.keys import (
    analyze_all_candidate_keys,
    analyze_all_normalization_collisions,
)
from argos.common.reports import write_report

cfg = PROJECT_CONFIG["profiling"]

# 1) Audit lossy normalization collisions explicitly. A collision is a
#    schema finding, not a reason to abort key analysis unless the candidate
#    key itself is ambiguous.
normalization_collisions = analyze_all_normalization_collisions(
    FILES,
    sample_size=cfg["csv_sample_size"],
    temp_directory=paths.root / ".cache" / "duckdb",
)

write_report(
    normalization_collisions,
    paths.reports / "profiling" / "column_normalization_collisions.parquet",
)

# 2) Analyze configured candidate keys. Unrelated normalization collisions
#    no longer block the analysis.
key_analysis = analyze_all_candidate_keys(
    FILES,
    DATASETS,
    sample_size=cfg["csv_sample_size"],
    temp_directory=paths.root / ".cache" / "duckdb",
)

write_report(
    key_analysis,
    paths.reports / "profiling" / "key_analysis.parquet",
)

display(normalization_collisions)
key_analysis


,dataset,normalized_column,raw_columns,collision_size,status
0,payments,fecha_de_emision,"[Fecha de emision, Fecha de emisión]",2,normalization_collision


,dataset,candidate_key,raw_columns,status,missing_columns,ambiguous_columns,row_count,rows_with_null_key,null_key_rate,distinct_nonnull_keys,nonnull_rows,uniqueness_ratio,duplicate_excess_rows,max_multiplicity,is_unique_when_nonnull
0,processes,id_del_proceso,[ID del Proceso],analyzed,None,None,9204423.0,0.0,0.000000e+00,8769182.0,9204423.0,0.952714,435241.0,8840.0,False
1,processes,referencia_del_proceso,[Referencia del Proceso],analyzed,None,None,9204423.0,0.0,0.000000e+00,6799136.0,9204423.0,0.738681,2405287.0,8840.0,False
2,processes,id_del_portafolio,[ID del Portafolio],analyzed,None,None,9204423.0,0.0,0.000000e+00,8540484.0,9204423.0,0.927867,663939.0,8840.0,False
3,contracts,id_contrato,[ID Contrato],analyzed,None,None,6066730.0,0.0,0.000000e+00,6066730.0,6066730.0,1.000000,0.0,1.0,True
4,contracts,referencia_del_contrato,[Referencia del Contrato],analyzed,None,None,6066730.0,1.0,1.648334e-07,4478838.0,6066729.0,0.738262,1587891.0,1869.0,False
5,bidders,id_procedimiento,[ID Procedimiento],analyzed,None,None,2316965.0,0.0,0.000000e+00,697182.0,2316965.0,0.300903,1619783.0,1620.0,False
6,bidders,id_procedimiento+id_proveedor,NaN,missing_columns,[id_proveedor],None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,bidders,id_procedimiento+codigo_proveedor,"[ID Procedimiento, Codigo Proveedor]",analyzed,None,None,2316965.0,0.0,0.000000e+00,2281440.0,2316965.0,0.984667,35525.0,154.0,False
8,suppliers,codigo,[Codigo],analyzed,None,None,1612550.0,0.0,0.000000e+00,1612550.0,1612550.0,1.000000,0.0,1.0,True
9,suppliers,nit,[NIT],analyzed,None,None,1612550.0,0.0,0.000000e+00,1605149.0,1612550.0,0.995410,7401.0,29.0,False


In [3]:
# Candidatos que podrían ser PK técnica cuando son no-null.
key_analysis.loc[
    key_analysis.get("is_unique_when_nonnull", False).fillna(False),
    ["dataset", "candidate_key", "row_count", "null_key_rate", "uniqueness_ratio", "max_multiplicity"]
]

,dataset,candidate_key,row_count,null_key_rate,uniqueness_ratio,max_multiplicity
3,contracts,id_contrato,6066730.0,0.0,1.0,1.0
8,suppliers,codigo,1612550.0,0.0,1.0,1.0


## Trabajo manual requerido
Completa `docs/data-dictionary/*.md` con una frase explícita del grain, por ejemplo:
`1 fila = 1 evento de modificación contractual`.

Si el análisis contradice la documentación fuente, conserva ambas evidencias y no corrijas silenciosamente.

## M1 remediation — duplicate structure

Uniqueness ratios alone do not tell us whether a repeated key is an exact duplicate or a genuinely versioned/multi-row entity. This diagnostic fingerprints the complete Bronze row for the two grains that block Silver design.


In [4]:
from argos.profiling.grain import analyze_duplicate_structure, top_duplicate_keys

grain_probes = [
    ("processes", ("id_del_proceso",)),
    ("additions", ("id_contrato", "fecharegistro", "tipo")),
]

grain_resolution = pd.DataFrame([
    analyze_duplicate_structure(
        dataset,
        FILES[dataset],
        key_columns,
        sample_size=cfg["csv_sample_size"],
        temp_directory=paths.root / ".cache" / "duckdb",
    )
    for dataset, key_columns in grain_probes
])

write_report(
    grain_resolution,
    paths.reports / "profiling" / "grain_resolution.parquet",
)

grain_resolution


,dataset,key,raw_columns,status,distinct_nonnull_keys,duplicate_key_count,rows_on_duplicate_keys,duplicate_excess_rows,exact_duplicate_excess_rows,exact_duplicate_key_count,conflicting_duplicate_key_count,max_key_multiplicity,max_payload_variants,grain_interpretation,silver_implication
0,processes,id_del_proceso,[ID del Proceso],analyzed,8769182,84558,519799,435241,136838,51564,32994,8840,7420,multiple_payload_variants_per_key,canonicalize_or_model_versioned_rows
1,additions,id_contrato+fecharegistro+tipo,"[ID_Contrato, FechaRegistro, Tipo]",analyzed,8636666,4783473,24525411,19741938,17771253,3052050,1731423,277,28,multiple_payload_variants_per_key,canonicalize_or_model_versioned_rows


In [5]:
duplicate_examples = pd.concat(
    [
        top_duplicate_keys(
            dataset,
            FILES[dataset],
            key_columns,
            sample_size=cfg["csv_sample_size"],
            limit=25,
            temp_directory=paths.root / ".cache" / "duckdb",
        )
        for dataset, key_columns in grain_probes
    ],
    ignore_index=True,
)

write_report(
    duplicate_examples,
    paths.reports / "profiling" / "duplicate_key_examples.parquet",
)

duplicate_examples


,dataset,key,id_del_proceso,rows,payload_variants,id_contrato,fecharegistro,tipo
0,processes,id_del_proceso,CO1.REQ.8303507,8840,1768,NaN,NaN,NaN
1,processes,id_del_proceso,CO1.REQ.2906406,7420,7420,NaN,NaN,NaN
2,processes,id_del_proceso,CO1.REQ.6931720,4805,961,NaN,NaN,NaN
3,processes,id_del_proceso,CO1.REQ.1241749,4190,4190,NaN,NaN,NaN
4,processes,id_del_proceso,CO1.REQ.6710358,3828,3828,NaN,NaN,NaN
5,processes,id_del_proceso,CO1.REQ.16902,3819,2010,NaN,NaN,NaN
6,processes,id_del_proceso,CO1.REQ.778356,3786,3786,NaN,NaN,NaN
7,processes,id_del_proceso,CO1.REQ.5154156,3690,3690,NaN,NaN,NaN
8,processes,id_del_proceso,CO1.REQ.7175364,2645,529,NaN,NaN,NaN
9,processes,id_del_proceso,CO1.REQ.6866412,2093,2093,NaN,NaN,NaN
